# Personal Budget Tracker

**Level:** Beginner

## Project description

You have a month of transaction data and need to summarize spending by
category. The goal is to find where money is going and identify unusually
large expenses.

## Skills practiced

- Loading transaction-style data
- Cleaning category labels
- Creating income and expense summaries
- Filtering unusual records


In [ ]:
import pandas as pd
import numpy as np
import secrets

practice_run_name = f"personal-budget-{secrets.token_hex(3)}"
print(f"Practice run: {practice_run_name}")

def make_transaction_data(row_count=40, seed=42):
    """Create a synthetic personal finance dataset for practice."""
    rng = np.random.default_rng(seed)
    expense_categories = np.array([
        "Groceries", "Utilities", "Entertainment", "Dining",
        "Housing", "Transport", "Education", "Health"
    ])
    merchants = {
        "Groceries": ["Metro Grocery", "Fresh Basket", "Corner Market"],
        "Utilities": ["City Power", "Water Works", "Mobile Plan"],
        "Entertainment": ["StreamBox", "Cinema House", "Game Store"],
        "Dining": ["Coffee Corner", "Noodle Bar", "Taco Stop"],
        "Housing": ["Rent", "Home Supplies"],
        "Transport": ["Train Pass", "Ride Share", "Fuel Station"],
        "Education": ["Book Store", "Online Course"],
        "Health": ["Doctor Office", "Pharmacy"],
    }

    dates = pd.date_range("2026-02-01", periods=28, freq="D")
    rows = []
    for _ in range(row_count):
        category = rng.choice(expense_categories)
        rows.append({
            "date": rng.choice(dates),
            "merchant": rng.choice(merchants[category]),
            "category": category,
            "amount": -round(float(rng.lognormal(mean=3.6, sigma=0.7)), 2),
        })

    rows.extend([
        {"date": pd.Timestamp("2026-02-01"), "merchant": "Paycheck", "category": "Income", "amount": 2400.00},
        {"date": pd.Timestamp("2026-02-25"), "merchant": "Paycheck", "category": "Income", "amount": 2400.00},
    ])

    transactions = pd.DataFrame(rows).sort_values("date").reset_index(drop=True)
    transactions.loc[transactions.index[1], "category"] = " groceries "
    return transactions

transactions = make_transaction_data(row_count=40, seed=42)
transactions


## Step 1: Clean category labels

Notice that one category has extra spaces and inconsistent capitalization.
Clean text early so grouped results are accurate.


In [ ]:
budget = transactions.copy()

budget["category"] = (
    budget["category"]
    .str.strip()
    .str.title()
)

budget


## Step 2: Create useful columns

Expenses are negative numbers in this dataset. We create a positive
`expense_amount` column so spending summaries are easier to read.


In [ ]:
budget["expense_amount"] = budget["amount"].where(budget["amount"] < 0, 0).abs()
budget["income_amount"] = budget["amount"].where(budget["amount"] > 0, 0)

budget


## Step 3: Summarize spending by category


In [ ]:
category_spending = (
    budget
    .groupby("category", as_index=False)
    .agg(
        total_spent=("expense_amount", "sum"),
        transaction_count=("merchant", "count"),
    )
    .sort_values("total_spent", ascending=False)
)

category_spending


In [ ]:
total_income = budget["income_amount"].sum()
total_expenses = budget["expense_amount"].sum()
net_cash_flow = total_income - total_expenses

pd.DataFrame({
    "metric": ["Total income", "Total expenses", "Net cash flow"],
    "amount": [total_income, total_expenses, net_cash_flow],
})


## Step 4: Find unusual expenses

For this project, define an unusual expense as any expense over $200.


In [ ]:
unusual_expenses = budget.loc[budget["expense_amount"] > 200]
unusual_expenses


## Final project notes

- Biggest spending category:
- Net cash flow:
- Unusual expense found:
- One budget change you recommend:


## Practice extension: Generate your own dataset

Create a larger or smaller month of transactions by changing `row_count` and
`seed`. Set `seed=None` for a randomized dataset each run. You can also edit
the merchant and category lists inside the function.


In [ ]:
my_transactions = make_transaction_data(row_count=75, seed=13)
my_transactions.head()
